# EDA — Labeled Misinformation Dataset (LMD)

Exploratory analysis of the combined dataset used for training Phase 1 models.
The LMD pulls from FakeNewsNet, LIAR, MultiFC and several other sources.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# load one of the configs — twitter_filtered is where we got best results
# adjust path as needed
DATA_PATH = '../data/processed/twitter_filtered/train.json'

if os.path.exists(DATA_PATH):
    df = pd.read_json(DATA_PATH)
else:
    # fallback to raw if processed not ready yet
    df = pd.read_csv('../data/raw/twitter_filtered.csv')

print(f'Loaded {len(df)} rows')
df.head()

In [ ]:
print(df.info())
print('\nLabel distribution:')
print(df['label'].value_counts())

## Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts()
labels_map = {0: 'Misinformation', 1: 'Factual'}
label_names = [labels_map[i] for i in counts.index]

axes[0].bar(label_names, counts.values, color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Class Distribution (Counts)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=label_names, autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[1].set_title('Class Distribution (%)')

plt.tight_layout()
plt.show()

print(f'\nClass balance ratio: {counts.min()/counts.max():.3f}')

## Text Length Analysis

Let's look at how text length differs between classes — misinformation often tends to be shorter/more punchy.

In [ ]:
# compute word count if not already there
if 'word_count' not in df.columns:
    df['word_count'] = df['content'].apply(lambda x: len(str(x).split()))
if 'char_count' not in df.columns:
    df['char_count'] = df['content'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, name, color in [(0, 'Misinformation', '#e74c3c'), (1, 'Factual', '#2ecc71')]:
    subset = df[df['label'] == label]['word_count']
    axes[0].hist(subset.clip(upper=200), bins=50, alpha=0.6, label=name, color=color)

axes[0].set_title('Word Count Distribution by Class')
axes[0].set_xlabel('Word Count')
axes[0].legend()

df_plot = df[['label', 'word_count', 'char_count']].copy()
df_plot['class'] = df_plot['label'].map({0: 'Misinformation', 1: 'Factual'})
sns.boxplot(data=df_plot, x='class', y='word_count', ax=axes[1],
            palette={'Misinformation': '#e74c3c', 'Factual': '#2ecc71'})
axes[1].set_title('Word Count by Class (Boxplot)')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

print('Mean word count by class:')
print(df.groupby('label')['word_count'].describe()[['mean', 'std', '50%']].round(1))

## Sentiment Analysis

Misinformation tends to be more emotionally charged — let's check.

In [ ]:
if 'sentiment_polarity' not in df.columns:
    print('Sentiment features not found — run preprocessing first')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    df_plot = df[['label', 'sentiment_polarity']].copy()
    df_plot['class'] = df_plot['label'].map({0: 'Misinformation', 1: 'Factual'})

    sns.violinplot(data=df_plot, x='class', y='sentiment_polarity', ax=axes[0],
                   palette={'Misinformation': '#e74c3c', 'Factual': '#2ecc71'})
    axes[0].set_title('Sentiment Polarity by Class')
    axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')

    # sentiment distribution histogram
    for label, name, color in [(0, 'Misinformation', '#e74c3c'), (1, 'Factual', '#2ecc71')]:
        subset = df[df['label'] == label]['sentiment_polarity']
        axes[1].hist(subset, bins=40, alpha=0.6, label=name, color=color)

    axes[1].set_title('Sentiment Polarity Distribution')
    axes[1].set_xlabel('Polarity (VADER compound score)')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print('Mean sentiment by class:')
    print(df.groupby('label')['sentiment_polarity'].mean().round(3))

## Feature Correlations

In [ ]:
important_features = ['sentiment_polarity', 'char_count', 'avg_word_length',
                       'verb_ratio', 'word_count', 'type_token_ratio', 'readability_score']

available = [f for f in important_features if f in df.columns]

if available:
    corr = df[available + ['label']].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                square=True, cbar_kws={'shrink': 0.8})
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()
else:
    print('Feature columns not found — run preprocessing with feature extraction')

## Top Words by Class

In [ ]:
stopwords = {'the','a','an','is','are','was','were','be','been','being','have',
             'has','had','do','does','did','will','would','could','should','may',
             'might','to','of','in','for','on','with','at','by','from','and',
             'or','but','not','this','that','it','i','he','she','they','we',
             'url','hashtag','mention','rt'}

def top_words(texts, n=20):
    words = []
    for t in texts:
        words.extend([w for w in str(t).lower().split() if w not in stopwords and len(w) > 2])
    return Counter(words).most_common(n)

misinfo_texts = df[df['label'] == 0]['content'].tolist()
factual_texts = df[df['label'] == 1]['content'].tolist()

top_misinfo = top_words(misinfo_texts)
top_factual = top_words(factual_texts)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

words_m, counts_m = zip(*top_misinfo)
axes[0].barh(words_m[::-1], counts_m[::-1], color='#e74c3c', alpha=0.8)
axes[0].set_title('Top 20 Words — Misinformation')
axes[0].set_xlabel('Frequency')

words_f, counts_f = zip(*top_factual)
axes[1].barh(words_f[::-1], counts_f[::-1], color='#2ecc71', alpha=0.8)
axes[1].set_title('Top 20 Words — Factual')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

## Key Findings

- **Class imbalance**: The dataset has varying degrees of imbalance depending on configuration. `twitter_filtered` is roughly balanced after preprocessing.
- **Text length**: Misinformation posts tend to be shorter on average — possibly due to the Twitter-style format.
- **Sentiment**: Misinformation posts show more negative and extreme sentiment polarity, consistent with emotionally charged content.
- **Vocabulary**: Certain political and emotionally loaded terms appear more frequently in misinformation.
- **Feature importance**: `sentiment_polarity` consistently ranks highest in feature importance analysis (confirmed later in SHAP).